# Conventionality Evaluator v2

**The Conventionality Evaluator** measures how explicit, literal, and straightforward a text's meaning is for students in grades 3–11. It returns:

* **complexity_score**: The conventionality complexity level (slightly_complex → exceedingly_complex).
* **conventionality_features**: Specific language features driving the complexity with direct quotes.
* **grade_context**: How the conventionality demands compare to expectations for the target grade.
* **instructional_insights**: Actionable pedagogical suggestions for scaffolding.
* **reasoning**: A synthesis of why the text fits the chosen complexity level.

---

**Handoff artifacts:**
- `prompts/conventionality/config.json` — evaluator spec (prompts, model config, output schema)
- `prompts/conventionality/corpus.csv` — labeled examples representing expected behavior

### Install & load packages

In [ ]:
%pip install -qU langchain-google-genai langchain pydantic textstat

In [ ]:
import getpass
import json
import os
from pathlib import Path
from typing import List, Literal

from dotenv import load_dotenv
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.chat import HumanMessagePromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from textstat import textstat as ts

### Load evaluator config

All evaluator configuration — prompts, model settings, and output schema — is defined in `config.json`. This is the handoff artifact shared with engineering.

In [ ]:
CONFIG_DIR = Path("prompts/conventionality")
config = json.loads((CONFIG_DIR / "config.json").read_text())

system_prompt = (CONFIG_DIR / config["prompts"]["system"]["path"]).read_text()
user_prompt = (CONFIG_DIR / config["prompts"]["user"]["path"]).read_text()

load_dotenv()
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API key: ")

model_cfg = config["model"]
model = ChatGoogleGenerativeAI(model=model_cfg["name"], temperature=model_cfg["temperature"])

### Define output schema

This Pydantic model mirrors `output_schema` in `config.json`. Engineering uses the JSON Schema to generate equivalent types in TypeScript, Python SDK, etc.

In [ ]:
class ConventionalityOutput(BaseModel):
    complexity_score: Literal[
        "slightly_complex",
        "moderately_complex",
        "very_complex",
        "exceedingly_complex"
    ] = Field(description="The conventionality complexity level of the text")
    reasoning: str = Field(description="A synthesis of why the text fits the chosen rubric level")
    conventionality_features: List[str] = Field(
        description="Specific language features driving complexity with direct quotes from the text"
    )
    grade_context: str = Field(
        description="How the conventionality demands compare to expectations for the target grade"
    )
    instructional_insights: str = Field(
        description="Actionable pedagogical suggestions for scaffolding the conventionality features"
    )

structured_model = model.with_structured_output(ConventionalityOutput)

### Define evaluation function

In [ ]:
def calculate_fk_score(text: str) -> float:
    return round(ts.flesch_kincaid_grade(text), 2)


def evaluate(text: str, grade: int) -> ConventionalityOutput:
    prompt = ChatPromptTemplate([
        SystemMessage(content=system_prompt),
        HumanMessagePromptTemplate.from_template(user_prompt),
    ])
    chain = prompt | structured_model
    return chain.invoke({
        "text": text,
        "grade": grade,
        "fk_score": calculate_fk_score(text),
    })

### Try it yourself

Replace `sample_text` and `grade` below to evaluate any passage.

In [ ]:
sample_text = """
"Well, then," said the teacher, "you may take your slate and go out behind the schoolhouse
for half an hour. Think of something to write about, and write the word on your slate.
Then try to tell what it is, what it is like, what it is good for, and what is done with it.
That is the way to write a composition." Henry took his slate and went out. Just behind the
schoolhouse was Mr. Finney's barn. Quite close to the barn was a garden. And in the garden,
Henry saw a turnip.
"""

result = evaluate(sample_text, grade=4)
display(result.model_dump())